# 030 Create Gold
Create the Gold layer in Silver


## Setup


### Imports


In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants


## Main


### Get the Silver table


In [ ]:
df = spark.table("hpi_silver")


### dim.date
Use SparkSQL to generate the date dimension (effectively in Silver), before pushing to Gold


In [ ]:
%%sql
CREATE OR REPLACE TABLE dim_date
AS
WITH date_bounds AS (
  SELECT
    make_date(year(min(Date)), 1, 1) AS start_date,
    make_date(year(max(Date)), 12, 31) AS end_date
  FROM lh_hpi.hpi_silver
),
date_sequence AS (
  SELECT explode(sequence(start_date, end_date, interval 1 day)) AS hpiDate
  FROM date_bounds
)
SELECT
  date_format(hpiDate, 'yyyyMMdd')            AS hpiDateId,
  hpiDate                                     AS fullDate,
  day(hpiDate)                                AS dayOfMonth,
  date_format(hpiDate, 'EEEE')                AS dayName,
  weekday(hpiDate) + 1                        AS dayOfWeek,        -- Monday=1
  weekofyear(hpiDate)                         AS weekOfYear,
  last_day(hpiDate)                           AS endOfMonthDate,
  date_format(hpiDate, 'MMMM')                AS monthName,
  month(hpiDate)                              AS monthNumber,
  quarter(hpiDate)                            AS quarterNumber,
  year(hpiDate)                               AS yearNumber,
  CASE WHEN weekday(hpiDate) IN (5,6) THEN true ELSE false END AS isWeekend,
  CASE WHEN day(hpiDate) = 1 THEN true ELSE false END AS isFirstOfMonth,
  CASE WHEN hpiDate = last_day(hpiDate) THEN true ELSE false END AS isLastOfMonth
FROM date_sequence
ORDER BY hpiDate;


### Bank Holidays


In [ ]:
#Add bank holidays
df_publicHolidays = spark.sql("SELECT * FROM lh_publicHolidays.publicHolidays WHERE countryOrRegion = 'England'")

df_publicHolidays.createOrReplaceTempView("publicHolidays")


### Working


In [ ]:
%%sql
SELECT
    d.*,
    NOT ISNULL( ph.date ) isBankHoliday,
    ph.holidayName
FROM dim_date d
    LEFT JOIN lh_publicHolidays.publicHolidays ph ON d.fullDate = ph.date
        AND ph.countryOrRegion = 'England'
WHERE yearNumber = 2025
--AND dayOfMonth = 1
--AND monthNumber = 1
--AND NOT ISNULL( ph.date ) = true
LIMIT 100;


In [ ]:
df_date = spark.sql("""
SELECT
    d.*,
    NOT ISNULL( ph.date ) isBankHoliday,
    ph.holidayName
FROM dim_date d
    LEFT JOIN lh_publicHolidays.publicHolidays ph ON d.fullDate = ph.date
        AND ph.countryOrRegion = 'England'
WHERE yearNumber = 2025
""")

df_date.write.mode("overwrite").synapsesql("dw_hpi.dim.date")


### dim.region


In [ ]:
%%sql
CREATE OR REPLACE TABLE dim_region
AS
SELECT
    ROW_NUMBER() OVER( ORDER BY RegionName ) AS regionId,
    RegionName AS regionName
FROM lh_hpi.hpi_silver
GROUP BY RegionName;


In [ ]:
df_region = spark.sql("SELECT * FROM dim_region")
df_region.write.mode("overwrite").synapsesql("dw_hpi.dim.region")


### dim.areaCode


In [ ]:
%%sql
CREATE OR REPLACE TABLE dim_areaCode
AS
SELECT
    ROW_NUMBER() OVER( ORDER BY s.AreaCode ) + 100 AS areaCodeId,
    r.regionId,
    s.AreaCode AS areaCode
FROM lh_hpi.hpi_silver s
    INNER JOIN lh_hpi.dim_region r ON s.RegionName = r.RegionName
GROUP BY s.AreaCode, r.regionId;


In [ ]:
df_areaCode = spark.sql("SELECT * FROM dim_areaCode")
df_areaCode.write.mode("overwrite").synapsesql("dw_hpi.dim.areaCode")


### fact


In [ ]:
%%sql
CREATE OR REPLACE TABLE fact_hpi
AS
SELECT
    d.hpiDateId,
    ac.areaCodeId,

    s.`AveragePrice` AS `AveragePrice`,
    s.`Index` AS `Index`,
    s.`IndexSA` AS `IndexSA`,
    s.`1m%Change` AS `1m%Change`,
    s.`12m%Change` AS `12m%Change`,
    s.`AveragePriceSA` AS `AveragePriceSA`,
    s.`SalesVolume` AS `SalesVolume`,
    s.`DetachedPrice` AS `DetachedPrice`,
    s.`DetachedIndex` AS `DetachedIndex`,
    s.`Detached1m%Change` AS `Detached1m%Change`,
    s.`Detached12m%Change` AS `Detached12m%Change`,
    s.`SemiDetachedPrice` AS `SemiDetachedPrice`,
    s.`SemiDetachedIndex` AS `SemiDetachedIndex`,
    s.`SemiDetached1m%Change` AS `SemiDetached1m%Change`,
    s.`SemiDetached12m%Change` AS `SemiDetached12m%Change`,
    s.`TerracedPrice` AS `TerracedPrice`,
    s.`TerracedIndex` AS `TerracedIndex`,
    s.`Terraced1m%Change` AS `Terraced1m%Change`,
    s.`Terraced12m%Change` AS `Terraced12m%Change`,
    s.`FlatPrice` AS `FlatPrice`,
    s.`FlatIndex` AS `FlatIndex`,
    s.`Flat1m%Change` AS `Flat1m%Change`,
    s.`Flat12m%Change` AS `Flat12m%Change`,
    s.`CashPrice` AS `CashPrice`,
    s.`CashIndex` AS `CashIndex`,
    s.`Cash1m%Change` AS `Cash1m%Change`,
    s.`Cash12m%Change` AS `Cash12m%Change`,
    s.`CashSalesVolume` AS `CashSalesVolume`,
    s.`MortgagePrice` AS `MortgagePrice`,
    s.`MortgageIndex` AS `MortgageIndex`,
    s.`Mortgage1m%Change` AS `Mortgage1m%Change`,
    s.`Mortgage12m%Change` AS `Mortgage12m%Change`,
    s.`MortgageSalesVolume` AS `MortgageSalesVolume`,
    s.`FTBPrice` AS `FTBPrice`,
    s.`FTBIndex` AS `FTBIndex`,
    s.`FTB1m%Change` AS `FTB1m%Change`,
    s.`FTB12m%Change` AS `FTB12m%Change`,
    s.`FOOPrice` AS `FOOPrice`,
    s.`FOOIndex` AS `FOOIndex`,
    s.`FOO1m%Change` AS `FOO1m%Change`,
    s.`FOO12m%Change` AS `FOO12m%Change`,
    s.`NewPrice` AS `NewPrice`,
    s.`NewIndex` AS `NewIndex`,
    s.`New1m%Change` AS `New1m%Change`,
    s.`New12m%Change` AS `New12m%Change`,
    s.`NewSalesVolume` AS `NewSalesVolume`,
    s.`OldPrice` AS `OldPrice`,
    s.`OldIndex` AS `OldIndex`,
    s.`Old1m%Change` AS `Old1m%Change`,
    s.`Old12m%Change` AS `Old12m%Change`,
    s.`OldSalesVolume` AS `OldSalesVolume`

FROM lh_hpi.hpi_silver s
    INNER JOIN dim_date d ON s.Date = d.fullDate
    INNER JOIN dim_areaCode ac ON s.AreaCode = ac.areaCode;


In [ ]:
df_factHpi = spark.sql("SELECT * FROM fact_hpi")
df_factHpi.write.mode("overwrite").synapsesql("dw_hpi.fact.Hpi")


In [ ]:
%%sql
MERGE 


In [ ]:
%%sql
MERGE INTO target_table AS tgt
USING source_table AS src
ON tgt.Id = src.Id
WHEN MATCHED THEN
  UPDATE SET
    tgt.Name      = src.Name,
    tgt.UpdatedAt = src.UpdatedAt
WHEN NOT MATCHED THEN
  INSERT (Id, Name, CreatedAt, UpdatedAt)
  VALUES (src.Id, src.Name, src.CreatedAt, src.UpdatedAt);


In [ ]:
%%sql
MERGE INTO dim_date AS tgt
USING (
  SELECT
    date_format(hpiDate, 'yyyyMMdd')            AS hpiDateId,
    hpiDate                                     AS fullDate,
    day(hpiDate)                                AS dayOfMonth,
    date_format(hpiDate, 'EEEE')                AS dayName,
    weekday(hpiDate) + 1                        AS dayOfWeek,
    weekofyear(hpiDate)                         AS weekOfYear,
    last_day(hpiDate)                           AS endOfMonthDate,
    date_format(hpiDate, 'MMMM')                AS monthName,
    month(hpiDate)                              AS monthNumber,
    quarter(hpiDate)                            AS quarterNumber,
    year(hpiDate)                               AS yearNumber,
    CASE WHEN weekday(hpiDate) IN (5,6) THEN true ELSE false END AS isWeekend,
    CASE WHEN day(hpiDate) = 1 THEN true ELSE false END AS isFirstOfMonth,
    CASE WHEN hpiDate = last_day(hpiDate) THEN true ELSE false END AS isLastOfMonth,
    NOT ISNULL(ph.date)                         AS isBankHoliday,
    ph.holidayName                              AS holidayName
  FROM (
      -- your date range logic
      WITH date_bounds AS (
        SELECT
          make_date(year(min(Date)), 1, 1)  AS start_date,
          make_date(year(max(Date)), 12, 31) AS end_date
        FROM lh_hpi.hpi_silver
      ),
      date_sequence AS (
        SELECT explode(sequence(start_date, end_date, interval 1 day)) AS hpiDate
        FROM date_bounds
      )
      SELECT hpiDate FROM date_sequence
  ) dsrc
  LEFT JOIN lh_publicHolidays.publicHolidays ph
    ON dsrc.hpiDate = ph.date
   AND ph.countryOrRegion = 'England'
) AS src
ON tgt.hpiDateId = src.hpiDateId
WHEN MATCHED THEN
  UPDATE SET
    tgt.fullDate       = src.fullDate,
    tgt.dayOfMonth     = src.dayOfMonth,
    tgt.dayName        = src.dayName,
    tgt.dayOfWeek      = src.dayOfWeek,
    tgt.weekOfYear     = src.weekOfYear,
    tgt.endOfMonthDate = src.endOfMonthDate,
    tgt.monthName      = src.monthName,
    tgt.monthNumber    = src.monthNumber,
    tgt.quarterNumber  = src.quarterNumber,
    tgt.yearNumber     = src.yearNumber,
    tgt.isWeekend      = src.isWeekend,
    tgt.isFirstOfMonth = src.isFirstOfMonth,
    tgt.isLastOfMonth  = src.isLastOfMonth,
    tgt.isBankHoliday  = src.isBankHoliday,
    tgt.holidayName    = src.holidayName
WHEN NOT MATCHED THEN
  INSERT (
    hpiDateId,
    fullDate,
    dayOfMonth,
    dayName,
    dayOfWeek,
    weekOfYear,
    endOfMonthDate,
    monthName,
    monthNumber,
    quarterNumber,
    yearNumber,
    isWeekend,
    isFirstOfMonth,
    isLastOfMonth,
    isBankHoliday,
    holidayName
  )
  VALUES (
    src.hpiDateId,
    src.fullDate,
    src.dayOfMonth,
    src.dayName,
    src.dayOfWeek,
    src.weekOfYear,
    src.endOfMonthDate,
    src.monthName,
    src.monthNumber,
    src.quarterNumber,
    src.yearNumber,
    src.isWeekend,
    src.isFirstOfMonth,
    src.isLastOfMonth,
    src.isBankHoliday,
    src.holidayName
  );


### median
